# Held-out rare cell types: can a method place a rare population it never saw?

**The question.** The rare-cell results so far show that scProto's metacells represent rare types well — but every one of those types was present during training. A reviewer can reasonably ask whether that is memorisation of populations the model was fitted on, or structure the method can impose on a rare population it meets for the first time.

**The design.** A *subset* of the rare cell types is deleted from the training data entirely — absent from the affinity graph, from Stage-1 pretraining, from Stage-2 training and from early stopping. Only some rare types are held out (never all), so the model still sees rare populations while training; what it has never seen is *these* ones. Then every arm is asked to place those cells through a frozen encoder, and is scored only on them.

This is the same protocol as `node_heldout_modularity.ipynb`, moved from a uniform random 20% of cells to the cells that actually make the claim: the rare ones.

| arm | Stage 1 | continuation | placing unseen cells |
|---|---|---|---|
| A1 | scVI (default) | Leiden on its latent | kNN in the scVI latent |
| A2 | scVI (default) | SEACells on its latent | kNN in the scVI latent |
| B  | scVI (default) | scProto Stage 2 on that encoder | native — frozen encoder → prototype argmax |

**Two things this fixes relative to the earlier node-holdout run.** First, all three arms are trained on the *same reduced data*; the node-holdout notebook had to document that its two-step baselines' encoders had seen every test cell's expression in their original full-data run. Second, SEACells is included rather than structurally excluded: it is not inductive, so it (and Leiden) get the standard extension — a held-out cell inherits the majority cluster of its k nearest training cells in their own scVI latent. That is the strongest reasonable version of these baselines, and it is part of the stated method, not a claim that they are inductive.

**What is measured**, all restricted to held-out cells, per batch, mean ± std, with a paired one-sided Wilcoxon against the scProto arm:
- **held-out modularity** — modularity on the untouched full-dataset graph, restricted to edges touching a held-out cell (`calc_modularity_per_batch`, the same statistic as Table 1)
- **recovery** — fraction of a held-out type's cells whose metacell has that type as its majority label (the paper's rare-cell recall)
- **homogeneity** — mean fraction of each held-out cell's metacell sharing its label (the rare table's homogeneity)
- **concentration** — share of a type's cells landing in its single most-used metacell: did they group together or scatter

**Code:** `interpretable_ssl/experiments/scvi_rare_holdout.py`, which reuses `scvi_stage2.py` for the training arms. Split, train-only h5ad, models and baselines all cache to Drive and reload — a disconnect costs only the epochs since the last evaluation.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Same install cell as the other rebuttal notebooks. Only needed once per fresh runtime.
# RESTART THE RUNTIME after this cell before running anything below.
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 142.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 133.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 131.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.2/711.2 kB 52.5 MB/s eta 0:00:00
   ━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 139.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 107.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
anndata 0.13.2 requires scipy!=1.17.0,>=1.14, but you have scipy 1.13.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 wh

In [ ]:
_checks = {
    'numpy': 'numpy', 'scipy': 'scipy', 'anndata': 'anndata', 'scanpy': 'scanpy',
    'scarches': 'scarches', 'scvi-tools': 'scvi', 'seacells': 'SEACells',
    'palantir': 'palantir', 'scib-metrics': 'scib_metrics', 'leidenalg': 'leidenalg',
    'python-igraph': 'igraph', 'umap-learn': 'umap', 'faiss-cpu': 'faiss',
}
_failed = []
for pkg_name, import_name in _checks.items():
    try:
        mod = __import__(import_name)
        print(f"  OK   {pkg_name:16s} (version {getattr(mod, '__version__', '?')})")
    except Exception as e:
        _failed.append(pkg_name)
        print(f"  FAIL {pkg_name:16s}: {type(e).__name__}: {e}")
print(f"\n{len(_failed)} failed: {_failed}" if _failed else f"\nAll {len(_checks)} packages import cleanly.")

  OK   numpy            (version 2.2.6)
  OK   scipy            (version 1.13.1)


/tmp/ipykernel_2883/2317117352.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(f"  OK   {pkg_name:16s} (version {getattr(mod, '__version__', '?')})")


  OK   anndata          (version 0.13.2)


/tmp/ipykernel_2883/2317117352.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"  OK   {pkg_name:16s} (version {getattr(mod, '__version__', '?')})")


  OK   scanpy           (version 1.12.3)


  FAIL scarches        : ImportError: cannot import name 'read' from 'anndata' (/usr/local/lib/python3.12/dist-packages/anndata/__init__.py)
  OK   scvi-tools       (version 1.5.0.post1)
  OK   seacells         (version 0.3.3)
  OK   palantir         (version 1.4.5)
  OK   scib-metrics     (version 0.6.0)
  OK   leidenalg        (version 0.12.0)
  OK   python-igraph    (version 1.0.0)
  OK   umap-learn       (version 0.5.12)
  OK   faiss-cpu        (version 1.14.1)

1 failed: ['scarches']


In [ ]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

In [ ]:
import os, json, glob
import numpy as np
import pandas as pd

from interpretable_ssl.experiments.scvi_rare_holdout import (
    run_all_datasets, run_rare_holdout_experiment, save_summary,
    select_heldout_rare_types, load_full_counts_adata,
)
from interpretable_ssl.datasets.dataset_configs import DATASETS

print("scvi_rare_holdout imports ready")

## Config

In [ ]:
DATASETS_TO_RUN = ['pancreas', 'lung', 'pbmc-immune']
dataset_display_names = {'pancreas': 'Pancreas', 'lung': 'Lung', 'pbmc-immune': 'Immune'}

# Which rare types to hold out. FRAC_TYPES=0.5 holds out the rarest half of the eligible
# rare types -- never all of them, so the model still sees rare populations in training.
FRAC_TYPES     = 0.5
MAX_TYPES      = None    # or an int to hold out exactly that many (rarest first)
MIN_CELLS      = 20      # skip types too small for per-batch metrics to mean anything
CELL_FRACTION  = 1.0     # 1.0 = the type is entirely absent from training
                         # <1.0 = severely under-represented instead of absent
MIN_TRAIN_PER_BATCH = 50 # never strip a batch below this -- keeps every held-out cell's
                         # batch one the model has seen, which the frozen-forward-pass
                         # assumption requires
SEED  = 0
KNN_K = 15               # neighbours used to place a held-out cell for Leiden/SEACells

# Training budgets -- same as the main scVI Stage-2 notebook.
SCVI_EPOCHS       = 50
SCVI_N_LATENT     = 10
STAGE2_MAX_EPOCHS = 20
EVAL_FREQ         = 3
PATIENCE          = 6
BATCH_SIZE        = 1024
UMAP_STEPS_PER_EPOCH = 500

for ds in DATASETS_TO_RUN:
    cfg = DATASETS[ds]
    print(f"{ds:<14} K={cfg['num_prototypes']:<5} label_key={cfg['label_key']:<18} batch_key={cfg.get('batch_key')}")

In [ ]:
# Preflight 1: the canonical ARBF graph of the FULL dataset must exist -- held-out
# modularity is scored on it, untouched. (The training graph is a different, smaller one
# built from the reduced data; that one is generated during the run.)
for ds in DATASETS_TO_RUN:
    found = sorted(glob.glob(os.path.join('./graphs', f'affinity_{ds}[0-9]*_ncomp50_kneighbors50_arbf.pkl')))
    print(f"  {'OK  ' if found else 'MISS'} {ds:<14} " + (os.path.basename(found[0]) if found else 'no cached graph'))

In [ ]:
# Preflight 2: which types WOULD be held out, and how many cells that removes.
# Cheap dry run -- no training. Check this before committing to a run: if a type is
# concentrated in one batch, or the counts look too small, adjust FRAC_TYPES/MIN_CELLS.
for ds in DATASETS_TO_RUN:
    print(f"\n=== {ds} ===")
    ad = load_full_counts_adata(ds)
    lk, bk = DATASETS[ds]['label_key'], DATASETS[ds].get('batch_key')
    chosen = select_heldout_rare_types(ad, lk, frac_types=FRAC_TYPES,
                                       max_types=MAX_TYPES, min_cells=MIN_CELLS, seed=SEED)
    sub = ad.obs[ad.obs[lk].astype(str).isin([str(c) for c in chosen])]
    print(f"  -> {len(sub)}/{ad.n_obs} cells ({len(sub)/ad.n_obs:.2%}) would be held out")
    display(pd.crosstab(sub[lk].astype(str), sub[bk].astype(str)))
    del ad

## Run

Per dataset: pick the held-out types, write the reduced training set, train scVI + both arms on it, then place and score the held-out cells. A dataset that fails is reported and skipped.

Each dataset prints its own summary as soon as it finishes — no need to wait for the whole sweep.

In [ ]:
df_rare_ho, df_sig, infos = run_all_datasets(
    DATASETS_TO_RUN,
    frac_types=FRAC_TYPES,
    max_types=MAX_TYPES,
    cell_fraction=CELL_FRACTION,
    min_train_per_batch=MIN_TRAIN_PER_BATCH,
    min_cells=MIN_CELLS,
    seed=SEED,
    knn_k=KNN_K,
    scvi_epochs=SCVI_EPOCHS,
    scvi_n_latent=SCVI_N_LATENT,
    stage2_max_epochs=STAGE2_MAX_EPOCHS,
    eval_freq=EVAL_FREQ,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    umap_steps_per_epoch=UMAP_STEPS_PER_EPOCH,
    skip_if_exists=True,
)
df_rare_ho

## What was held out

In [ ]:
rows = [{
    'dataset': dataset_display_names.get(ds, ds),
    'held-out types': ', '.join(i['heldout_types']),
    'held-out cells': i['n_heldout_cells'],
    'train cells': i['n_train_cells'],
    'K': i['K'],
} for ds, i in infos.items()]
pd.DataFrame(rows).set_index('dataset')

## Results on the held-out cells

`heldout_modularity` is the generalization number: modularity on the untouched full-dataset graph, restricted to edges touching a cell no arm ever saw. `full_graph_modularity` is the same assignment scored on the whole graph, for reference against Table 1.

In [ ]:
def fmt(m, s):
    return None if pd.isna(m) else f"{m:.3f} ± {s:.3f}"

disp = df_rare_ho.copy()
disp['dataset'] = disp['dataset'].map(lambda d: dataset_display_names.get(d, d))
for base in ['heldout_modularity', 'full_graph_modularity', 'recovery', 'homogeneity', 'concentration']:
    disp[base] = disp.apply(lambda r: fmt(r[f'{base}_mean'], r[f'{base}_std']), axis=1)

display(disp.set_index(['dataset', 'method'])[
    ['heldout_modularity', 'recovery', 'homogeneity', 'concentration',
     'full_graph_modularity', 'rare_edge_same_cluster_rate']
])

## Significance

Paired one-sided Wilcoxon across batches (scProto Stage 2 > baseline), Bonferroni-corrected per dataset and metric. Batches are the pairing unit: which cell types are locally rare is a property of the batch, not of the method, so every arm is scored on the identical set of batches.

In [ ]:
for metric in df_sig['metric'].unique():
    print(f"=== {metric} ===")
    sub = df_sig[df_sig['metric'] == metric].copy()
    sub['dataset'] = sub['dataset'].map(lambda d: dataset_display_names.get(d, d))
    sub['cell'] = sub.apply(
        lambda r: f"{r['mean']:.3f}±{r['std']:.3f} (n={r['n']}) [ref]"
        if pd.isna(r.get('p_adj'))
        else f"{r['mean']:.3f}±{r['std']:.3f} (n={r['n']}, wins={r.get('n_wins','?')}/{r['n']})  "
             f"{r.get('sig','?')}  p_adj={r.get('p_adj', float('nan')):.3g}",
        axis=1)
    display(sub.pivot(index='method', columns='dataset', values='cell'))

df_sig

## Save summary

In [ ]:
out_dir = '/content/drive/MyDrive/codes/interpretable-prototype/neurips_manuscript/rebuttle/experiment-results'
save_summary(df_rare_ho, df_sig, infos, out_dir)